In [1]:
# --- replication package paths (auto-inserted) ---
from pathlib import Path

# Resolve the package root whether run from notebooks/ or the root.
_here = Path.cwd()
ROOT = _here if (_here / "data").exists() else _here.parent

DATA    = ROOT / "data"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

assert DATA.exists(), f"data folder not found: {DATA}"


In [2]:
"""
H1 / H1a Robustness: Zero-Truncated Negative Binomial  (v2)
--------------------------------------------------------------------------
CHANGES FROM v1
  * Reports converged status explicitly for every fit -- a non-converged
    ZTNB fit produces unreliable coefficients/SEs, so this must be checked
    before the results are reported.
  * Warm-starts ZTNB from the ordinary NB solution, which usually resolves
    the convergence failures ZTNB is prone to on its own.
  * REMOVED the AIC column. Ordinary NB and zero-truncated NB have
    different likelihoods over different supports, so their AICs are NOT
    comparable. The v1 script printed both, which was misleading.
  * Adds build_h1a_table() so the H1a repo-level table can be regenerated
    without re-running the whole H1a pipeline.

WHY THIS ANALYSIS EXISTS
The H1/H1a repository table only contains repositories observed as a
provenance source at least once, so reuse_count >= 1 by construction
(N=17,368, min=1, zero zeros). Ordinary NB places probability mass on
y=0, which cannot occur here, so its likelihood is misspecified for a
positive-only outcome. This is distinct from zero-INFLATION (too many
zeros), which this data cannot exhibit, and is not addressed by the
earlier Poisson-vs-NB comparison, since both share the same support.

REQUIREMENTS: statsmodels >= 0.13
"""

import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.discrete.truncated_model import TruncatedLFNegativeBinomialP

H1_PATH = RESULTS / "h1_analysis_dataframe.csv"
H1A_PATH = RESULTS / "h1a_analysis_dataframe.csv"

GITHUB_URL_PATTERN = r'(https://github\.com/[^/]+/[^/]+)/.*'


def _converged(model):
    """statsmodels stores convergence in mle_retvals; be defensive about it."""
    try:
        return bool(model.mle_retvals.get("converged", False))
    except Exception:
        return None


def fit_and_compare(df, predictor, label):
    df = df.copy()
    df[predictor] = df[predictor].astype(int)
    df["age_std"] = (df["age_days"] - df["age_days"].mean()) / df["age_days"].std()
    # Standardize log-age. ZTNB's Hessian inversion fails on raw log_age
    # (mean~7.3, sd~0.84): the intercept and slope become near-collinear and
    # the information matrix is ill-conditioned. Centering/scaling is a purely
    # numerical fix -- it changes the intercept and the age coefficient's
    # scale, but leaves the predictor of interest (and its p-value) unchanged.
    _la = np.log(df["age_days"] + 1)
    df["log_age"] = (_la - _la.mean()) / _la.std()

    y = df["reuse_count"].astype(float)
    print(f"\n{'='*72}\n{label}\n{'='*72}")
    print(f"N = {len(df):,}   min(y) = {y.min():.0f}   zeros = {(y == 0).sum()}")
    if y.min() >= 1:
        print(">> Confirmed zero-truncated: y >= 1 by construction.")
    else:
        n0 = int((y == 0).sum())
        print(f"\n  !! ABORT: {n0} observations have reuse_count = 0.")
        print("  !! A zero-truncated model assumes zeros are impossible, so it")
        print("  !! cannot be fitted to this table. The table is wrong -- it should")
        print("  !! contain only repositories observed as a provenance source.")
        print("  !! Fix the table (check the sink_repository_url notna() filter)")
        print("  !! and re-run. Do NOT report results from this table.\n")
        return

    for age_var in ["age_std", "log_age"]:
        X = sm.add_constant(df[[predictor, age_var]])
        print(f"\n--- Age specification: {age_var} ---")

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)  # benign log excursions
            nb = sm.NegativeBinomial(y, X).fit(disp=False, maxiter=500)

            # Warm-start ZTNB from the NB solution; fall back to a cold,
            # derivative-free start if that fails.
            try:
                ztnb = TruncatedLFNegativeBinomialP(y, X, truncation=0).fit(
                    start_params=nb.params.values, disp=False, maxiter=2000,
                    method="bfgs")
            except Exception as e:
                print(f"  [warm start failed: {e}; retrying cold with Nelder-Mead]")
                ztnb = TruncatedLFNegativeBinomialP(y, X, truncation=0).fit(
                    disp=False, maxiter=2000, method="nm")

            # If the Hessian could not be inverted (NaN SEs), retry with a
            # derivative-free optimizer before declaring the fit unusable.
            if np.isnan(ztnb.bse[predictor]):
                print("  [NaN SE from BFGS; retrying with Nelder-Mead]")
                try:
                    ztnb = TruncatedLFNegativeBinomialP(y, X, truncation=0).fit(
                        start_params=nb.params.values, disp=False,
                        maxiter=5000, method="nm")
                except Exception as e2:
                    print(f"  [Nelder-Mead retry also failed: {e2}]")

        rows = []
        for name, model in [("Ordinary NB", nb), ("Zero-truncated NB", ztnb)]:
            coef, se, p = model.params[predictor], model.bse[predictor], model.pvalues[predictor]
            rows.append({
                "model": name,
                "converged": _converged(model),
                "coef": round(coef, 4),
                "SE": round(se, 4),
                "IRR": round(np.exp(coef), 4),
                "CI_low": round(np.exp(coef - 1.96 * se), 4),
                "CI_high": round(np.exp(coef + 1.96 * se), 4),
                "p": f"{p:.4g}",
            })
        print(pd.DataFrame(rows).to_string(index=False))
        print("  (AIC deliberately omitted: NB and ZTNB likelihoods are not comparable.)")

        if _converged(ztnb) is False:
            print("  !! ZTNB DID NOT CONVERGE -- do not report these estimates. "
                  "Try method='nm', more maxiter, or rescale age.")
            continue

        p_nb, p_zt = nb.pvalues[predictor], ztnb.pvalues[predictor]
        b_nb, b_zt = nb.params[predictor], ztnb.params[predictor]

        # A NaN p-value or SE means the fit failed (e.g. Hessian inversion
        # failure). Treat that as a failure, never as agreement -- comparing
        # NaN < .05 silently evaluates False and can fake a match.
        if np.isnan(p_zt) or np.isnan(ztnb.bse[predictor]):
            print("  !! ZTNB produced NaN SE/p (Hessian inversion likely failed).")
            print("  !! No valid comparison possible. Do not report this row.")
            continue

        same_dir = np.sign(b_nb) == np.sign(b_zt)
        same_sig = (p_nb < .05) == (p_zt < .05)
        ratio = abs(b_zt / b_nb) if b_nb != 0 else float("inf")
        big_shift = ratio > 2 or ratio < 0.5

        print(f"  Direction match: {same_dir} | alpha=.05 verdict match: {same_sig} "
              f"| |coef| ratio ZTNB/NB: {ratio:.2f}")
        if same_dir and same_sig and not big_shift:
            print("  => Conclusion UNCHANGED under truncation.")
        elif same_dir and same_sig and big_shift:
            print("  => CAUTION: same significance verdict, but effect MAGNITUDE "
                  "shifts substantially. Report both estimates explicitly.")
        else:
            print("  => WARNING: conclusion CHANGES; report the truncated model "
                  "as primary.")


def build_h1a_table(relationships_csv="license_analysis_results_processed.csv",
                    dsr_engine_path="DSR_engine.py"):
    """Regenerate the H1a repo-level table (Permissive/Reciprocal only)."""
    ns = {}
    exec(open(dsr_engine_path).read(), ns)
    license_mapping, get_license_group = ns["license_mapping"], ns["get_license_group"]

    def classify(lic):
        if pd.isna(lic):
            return "Unresolved"
        spdx = license_mapping.get(str(lic).strip())
        if spdx is None:
            return "Unresolved"
        g = get_license_group(spdx)
        return "Permissive" if g == "Permissive" else (
            "Reciprocal" if g in ("Weak Copyleft", "Strong Copyleft") else "Unresolved")

    df = pd.read_csv(relationships_csv)
    df = df.drop_duplicates(subset=["method_hash", "source_repository_url", "sink_repository_url"])
    df = df[df["violation_lcd_category"] != 0]
    # NOTE: notna() is required. Without it, NaN sink URLs survive
    # (astype(str) renders NaN as "nan", which does not start with "N/A"),
    # producing repos with reuse_count = 0 and breaking the zero-truncation
    # premise of this entire analysis.
    df = df[df["sink_repository_url"].notna()]
    df = df[~df["sink_repository_url"].astype(str).str.startswith("N/A")]

    df["src"] = df["source_repository_url"].astype(str).str.extract(GITHUB_URL_PATTERN, expand=False)
    df["snk"] = df["sink_repository_url"].astype(str).str.extract(GITHUB_URL_PATTERN, expand=False)
    df = df.dropna(subset=["src"])
    df["license_family"] = df["source_file_license"].apply(classify)

    reuse = df.dropna(subset=["snk"]).groupby("src")["snk"].nunique()
    fam = df.groupby("src")["license_family"].agg(
        lambda s: s.mode().iloc[0] if not s.mode().empty else "Unresolved")
    ref = df["source_version"].max()
    age = (ref - df.groupby("src")["source_version"].min()) / (1000 * 60 * 60 * 24)

    out = pd.DataFrame({"reuse_count": reuse, "license_family": fam, "age_days": age})
    out = out.reset_index().rename(columns={"src": "repo_url"})
    out["reuse_count"] = out["reuse_count"].fillna(0).astype(int)
    out = out[out["license_family"].isin(["Permissive", "Reciprocal"])].dropna(subset=["age_days"])
    out["is_permissive"] = (out["license_family"] == "Permissive").astype(int)

    out.to_csv(H1A_PATH, index=False)
    n, npe, nre = len(out), int((out['license_family']=='Permissive').sum()), \
                  int((out['license_family']=='Reciprocal').sum())
    print(f"[built] {H1A_PATH}: N={n} ({npe} Permissive, {nre} Reciprocal)")
    if (n, npe, nre) != (11252, 9035, 2217):
        print(f"  !! MISMATCH vs the published H1a table (11,252 / 9,035 / 2,217).")
        print(f"  !! This reconstruction does not reproduce your pipeline. Export the")
        print(f"  !! table directly from your H1a script instead of rebuilding it here.")
    else:
        print("  Matches the published H1a table.")
    if (out["reuse_count"] == 0).any():
        print(f"  !! {int((out['reuse_count']==0).sum())} rows have reuse_count = 0 -- "
              f"the table is wrong; zero-truncation does not hold.")
    return out


if __name__ == "__main__":
    fit_and_compare(pd.read_csv(H1_PATH), "has_license",
                    "H1: reuse_count ~ has_license + age")

    try:
        h1a = pd.read_csv(H1A_PATH)
    except FileNotFoundError:
        print(f"\n[{H1A_PATH} not found -- building it]")
        h1a = build_h1a_table()

    if "is_permissive" not in h1a.columns:
        h1a["is_permissive"] = (h1a["license_family"] == "Permissive").astype(int)
    fit_and_compare(h1a, "is_permissive", "H1a: reuse_count ~ is_permissive + age")


H1: reuse_count ~ has_license + age
N = 17,368   min(y) = 1   zeros = 0
>> Confirmed zero-truncated: y >= 1 by construction.

--- Age specification: age_std ---
            model  converged   coef     SE    IRR  CI_low  CI_high      p
      Ordinary NB       True 0.0140 0.0144 1.0141  0.9858   1.0431 0.3322
Zero-truncated NB       True 0.0361 0.0334 1.0367  0.9711   1.1068 0.2798
  (AIC deliberately omitted: NB and ZTNB likelihoods are not comparable.)
  Direction match: True | alpha=.05 verdict match: True | |coef| ratio ZTNB/NB: 2.58
  => CAUTION: same significance verdict, but effect MAGNITUDE shifts substantially. Report both estimates explicitly.

--- Age specification: log_age ---
            model  converged   coef     SE    IRR  CI_low  CI_high      p
      Ordinary NB       True 0.0124 0.0142 1.0125  0.9846   1.0411 0.3844
Zero-truncated NB       True 0.0021 0.0335 1.0021  0.9384   1.0702 0.9496
  (AIC deliberately omitted: NB and ZTNB likelihoods are not comparable.)
  Direc

In [3]:
"""
Average Marginal Effects: Ordinary NB vs Zero-Truncated NB
--------------------------------------------------------------------------
WHY THIS IS NEEDED

The ZTNB run shows is_permissive significant under both age
specifications, whereas ordinary NB showed it significant only under
linear age. That is a genuine change in the significance verdict.

But the COEFFICIENTS are not directly comparable between the two models:

  * Ordinary NB:  beta acts on mu = E[y]
  * Zero-truncated NB: beta acts on the LATENT mu of the untruncated
    distribution, while the observable quantity is E[y | y >= 1]
    = mu / (1 - P(Y=0)).

Truncation compresses the observable range, so a given change in observed
counts requires a larger beta than in an untruncated model. Reporting
"IRR 1.05 -> 1.21" as if the effect quadrupled would therefore be wrong.

This script computes the AVERAGE MARGINAL EFFECT (AME) of is_permissive
on the OBSERVED count scale for both models, which IS comparable:

    AME = mean_i [ Ehat(y | x_i, permissive=1) - Ehat(y | x_i, permissive=0) ]

reported both in absolute counts and as a percentage of the baseline
predicted mean. Use these numbers -- not the raw IRRs -- when stating
how much more (or less) permissively licensed repositories are reused.

NB2 parameterisation used here:
    P(Y=0) = (1 + alpha*mu)^(-1/alpha)
    E[Y | Y>=1] = mu / (1 - P(Y=0))
"""

import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.discrete.truncated_model import TruncatedLFNegativeBinomialP

H1A_PATH = RESULTS / "h1a_analysis_dataframe.csv"
PREDICTOR = "is_permissive"
N_BOOT = 200          # bootstrap reps for the AME confidence interval
RNG_SEED = 42


def _mu(params, X):
    """Conditional latent mean exp(X @ beta); params excludes alpha."""
    return np.exp(X @ params)


def _e_y_given_positive(mu, alpha):
    """E[Y | Y >= 1] for NB2."""
    p0 = np.power(1.0 + alpha * mu, -1.0 / alpha)
    return mu / (1.0 - p0)


def ame_for_model(model, X_df, kind):
    """AME of PREDICTOR on the observed count scale.

    kind='nb'   -> observed scale is E[y]      = mu
    kind='ztnb' -> observed scale is E[y|y>=1] = mu / (1 - P(Y=0))
    """
    beta = model.params.drop("alpha").values if "alpha" in model.params.index \
        else model.params.values[:-1]
    alpha = float(model.params["alpha"]) if "alpha" in model.params.index \
        else float(model.params.values[-1])

    X1 = X_df.copy(); X1[PREDICTOR] = 1
    X0 = X_df.copy(); X0[PREDICTOR] = 0
    mu1, mu0 = _mu(beta, X1.values), _mu(beta, X0.values)

    if kind == "nb":
        y1, y0 = mu1, mu0
    else:
        y1 = _e_y_given_positive(mu1, alpha)
        y0 = _e_y_given_positive(mu0, alpha)

    return float(np.mean(y1 - y0)), float(np.mean(y0))


def fit_pair(df, age_var):
    y = df["reuse_count"].astype(float)
    X_df = sm.add_constant(df[[PREDICTOR, age_var]])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        nb = sm.NegativeBinomial(y, X_df).fit(disp=False, maxiter=500)
        ztnb = TruncatedLFNegativeBinomialP(y, X_df, truncation=0).fit(
            start_params=nb.params.values, disp=False, maxiter=2000, method="bfgs")
        if np.isnan(ztnb.bse[PREDICTOR]):
            ztnb = TruncatedLFNegativeBinomialP(y, X_df, truncation=0).fit(
                start_params=nb.params.values, disp=False, maxiter=5000, method="nm")
    return nb, ztnb, X_df


def bootstrap_all_ames(df, n_boot=N_BOOT, seed=RNG_SEED, verbose=True):
    """Percentile bootstrap CIs for ALL four AMEs in a single pass.

    The earlier version refit both models separately for each of the four
    (spec, model) combinations, doing 4x the necessary work. This fits each
    resample once per age spec and harvests both models' AMEs from it.
    """
    rng = np.random.default_rng(seed)
    n = len(df)
    acc = {("age_std", "nb"): [], ("age_std", "ztnb"): [],
           ("log_age", "nb"): [], ("log_age", "ztnb"): []}

    for b in range(n_boot):
        if verbose and b % 25 == 0:
            print(f"    bootstrap {b}/{n_boot}...", flush=True)
        samp = df.iloc[rng.integers(0, n, n)]
        for age_var in ["age_std", "log_age"]:
            try:
                nb, ztnb, X_df = fit_pair(samp, age_var)
                for kind, model in [("nb", nb), ("ztnb", ztnb)]:
                    a, _ = ame_for_model(model, X_df, kind)
                    if np.isfinite(a):
                        acc[(age_var, kind)].append(a)
            except Exception:
                continue

    out = {}
    for key, vals in acc.items():
        if len(vals) < 30:
            out[key] = (np.nan, np.nan, len(vals))
        else:
            lo, hi = np.quantile(vals, [0.025, 0.975])
            out[key] = (float(lo), float(hi), len(vals))
    return out


def run(path=H1A_PATH, do_bootstrap=True):
    df = pd.read_csv(path)
    df[PREDICTOR] = (df["license_family"] == "Permissive").astype(int) \
        if PREDICTOR not in df.columns else df[PREDICTOR].astype(int)
    df["age_std"] = (df["age_days"] - df["age_days"].mean()) / df["age_days"].std()
    _la = np.log(df["age_days"] + 1)
    df["log_age"] = (_la - _la.mean()) / _la.std()

    print(f"N = {len(df):,}   zeros = {int((df['reuse_count']==0).sum())}\n")

    boot = {}
    if do_bootstrap:
        print(f"  Running {N_BOOT} bootstrap reps (this takes a few minutes)...")
        boot = bootstrap_all_ames(df)

    rows = []
    for age_var in ["age_std", "log_age"]:
        nb, ztnb, X_df = fit_pair(df, age_var)
        for name, model, kind in [("Ordinary NB", nb, "nb"),
                                  ("Zero-truncated NB", ztnb, "ztnb")]:
            ame, base = ame_for_model(model, X_df, kind)
            lo, hi, nrep = boot.get((age_var, kind), (np.nan, np.nan, np.nan))
            rows.append({
                "age spec": age_var,
                "model": name,
                "coef": round(float(model.params[PREDICTOR]), 4),
                "p": f"{float(model.pvalues[PREDICTOR]):.4g}",
                "baseline E[y]": round(base, 3),
                "AME (counts)": round(ame, 4),
                "AME (% of baseline)": round(100 * ame / base, 2),
                "AME 95% CI": (f"[{lo:.4f}, {hi:.4f}]"
                               if np.isfinite(lo) else "n/a"),
            })

    out = pd.DataFrame(rows)
    print(out.to_string(index=False))
    print("\nInterpretation notes:")
    print(" * Compare models on AME (counts / % of baseline), NOT on coef or IRR.")
    print("   The ZTNB coefficient acts on the latent untruncated mean, so its")
    print("   IRR is not on the same scale as the ordinary NB IRR.")
    print(" * 'baseline E[y]' is the mean predicted count with is_permissive=0,")
    print("   on each model's own observed scale (E[y] for NB, E[y|y>=1] for ZTNB).")
    out.to_csv(RESULTS / "h1a_ame_comparison.csv", index=False)
    print("\nSaved: h1a_ame_comparison.csv")
    return out


if __name__ == "__main__":
    run(do_bootstrap=False)

N = 11,252   zeros = 0

age spec             model   coef         p  baseline E[y]  AME (counts)  AME (% of baseline) AME 95% CI
 age_std       Ordinary NB 0.0462   0.02938          1.880        0.0890                 4.73        n/a
 age_std Zero-truncated NB 0.1893 0.0001805          2.783        0.3197                11.49        n/a
 log_age       Ordinary NB 0.0199    0.3402          1.890        0.0380                 2.01        n/a
 log_age Zero-truncated NB 0.1149   0.02233          1.844        0.0879                 4.76        n/a

Interpretation notes:
 * Compare models on AME (counts / % of baseline), NOT on coef or IRR.
   The ZTNB coefficient acts on the latent untruncated mean, so its
   IRR is not on the same scale as the ordinary NB IRR.
 * 'baseline E[y]' is the mean predicted count with is_permissive=0,
   on each model's own observed scale (E[y] for NB, E[y|y>=1] for ZTNB).

Saved: h1a_ame_comparison.csv


In [4]:
"""
Calibration check: are the ZTNB fits actually well-fitted?
--------------------------------------------------------------------------
MOTIVATION

The AME run produced an inconsistency worth resolving before reporting:

    ZTNB, linear age : baseline E[y|y>=1] = 2.783,  AME = 11.5%
    ZTNB, log age    : baseline E[y|y>=1] = 1.844,  AME =  4.8%

For the same model on the same data, the average predicted count should
not depend much on how the age covariate is parameterised. A baseline of
2.783 is also ~39% above the observed Reciprocal-group mean of 2.004,
which suggests the linear-age fit may be poorly calibrated or resting at
a bad optimum, despite reporting converged=True.

WHAT THIS CHECKS

  1. Mean predicted count vs mean observed count (should be close).
  2. Predicted vs observed across the count distribution (1, 2, 3, 4, 5+).
  3. Log-likelihood at the reported optimum vs a Nelder-Mead refit from
     the same start, to detect a bad local optimum.

If the linear-age ZTNB fit is miscalibrated and the log-age fit is not,
report the log-age estimate as primary and say so explicitly.
"""

import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.discrete.truncated_model import TruncatedLFNegativeBinomialP

H1A_PATH = RESULTS / "h1a_analysis_dataframe.csv"
PREDICTOR = "is_permissive"


def e_y_given_positive(mu, alpha):
    p0 = np.power(1.0 + alpha * mu, -1.0 / alpha)
    return mu / (1.0 - p0)


def prob_k_ztnb(k, mu, alpha):
    """P(Y=k | Y>=1) for NB2, k >= 1, vectorised over mu."""
    r = 1.0 / alpha
    from scipy.special import gammaln
    logp = (gammaln(k + r) - gammaln(r) - gammaln(k + 1)
            + r * np.log(r / (r + mu)) + k * np.log(mu / (r + mu)))
    p0 = np.power(1.0 + alpha * mu, -1.0 / alpha)
    return np.exp(logp) / (1.0 - p0)


def check(df, age_var):
    y = df["reuse_count"].astype(float).values
    X_df = sm.add_constant(df[[PREDICTOR, age_var]])

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        nb = sm.NegativeBinomial(pd.Series(y), X_df).fit(disp=False, maxiter=500)
        zt_bfgs = TruncatedLFNegativeBinomialP(pd.Series(y), X_df, truncation=0).fit(
            start_params=nb.params.values, disp=False, maxiter=2000, method="bfgs")
        zt_nm = TruncatedLFNegativeBinomialP(pd.Series(y), X_df, truncation=0).fit(
            start_params=nb.params.values, disp=False, maxiter=8000, method="nm")

    print(f"\n{'='*68}\nAge spec: {age_var}\n{'='*68}")
    print(f"Observed mean reuse_count: {y.mean():.4f}   n = {len(y):,}")

    for name, m in [("ZTNB (BFGS)", zt_bfgs), ("ZTNB (Nelder-Mead)", zt_nm)]:
        beta = m.params.values[:-1]
        alpha = float(m.params.values[-1])
        mu = np.exp(X_df.values @ beta)
        pred = e_y_given_positive(mu, alpha)

        print(f"\n  {name}")
        print(f"    log-likelihood      : {m.llf:.3f}")
        print(f"    converged           : {m.mle_retvals.get('converged', '?')}")
        print(f"    mean predicted E[y] : {pred.mean():.4f}   "
              f"(observed {y.mean():.4f}, ratio {pred.mean()/y.mean():.3f})")
        print(f"    coef[{PREDICTOR}]  : {m.params[PREDICTOR]:.4f}   "
              f"p = {m.pvalues[PREDICTOR]:.4g}")
        print(f"    alpha               : {alpha:.4f}")

        # distribution check
        print(f"    {'k':>4} {'observed':>10} {'predicted':>10}")
        for k in [1, 2, 3, 4]:
            obs = float((y == k).mean())
            prd = float(prob_k_ztnb(k, mu, alpha).mean())
            print(f"    {k:>4} {obs:>10.4f} {prd:>10.4f}")
        obs5 = float((y >= 5).mean())
        prd5 = 1.0 - sum(float(prob_k_ztnb(k, mu, alpha).mean()) for k in [1, 2, 3, 4])
        print(f"    {'5+':>4} {obs5:>10.4f} {prd5:>10.4f}")

    if zt_nm.llf > zt_bfgs.llf + 1e-3:
        print(f"\n  !! Nelder-Mead found a HIGHER log-likelihood "
              f"({zt_nm.llf:.3f} > {zt_bfgs.llf:.3f}).")
        print(f"  !! The BFGS fit was at a worse optimum -- use the NM estimates.")
    else:
        print(f"\n  BFGS optimum is at least as good as Nelder-Mead "
              f"({zt_bfgs.llf:.3f} vs {zt_nm.llf:.3f}).")


if __name__ == "__main__":
    df = pd.read_csv(H1A_PATH)
    if PREDICTOR not in df.columns:
        df[PREDICTOR] = (df["license_family"] == "Permissive").astype(int)
    df["age_std"] = (df["age_days"] - df["age_days"].mean()) / df["age_days"].std()
    _la = np.log(df["age_days"] + 1)
    df["log_age"] = (_la - _la.mean()) / _la.std()

    for spec in ["age_std", "log_age"]:
        check(df, spec)


Age spec: age_std
Observed mean reuse_count: 1.9220   n = 11,252

  ZTNB (BFGS)
    log-likelihood      : -13508.700
    converged           : True
    mean predicted E[y] : 2.9673   (observed 1.9220, ratio 1.544)
    coef[is_permissive]  : 0.1893   p = 0.0001805
    alpha               : 4607.3756
       k   observed  predicted
       1     0.6567     0.6149
       2     0.1752     0.1922
       3     0.0662     0.0828
       4     0.0352     0.0415
      5+     0.0667     0.0685

  ZTNB (Nelder-Mead)
    log-likelihood      : -13508.562
    converged           : True
    mean predicted E[y] : 2.9676   (observed 1.9220, ratio 1.544)
    coef[is_permissive]  : 0.1891   p = 0.0002127
    alpha               : 325416.7350
       k   observed  predicted
       1     0.6567     0.6149
       2     0.1752     0.1922
       3     0.0662     0.0828
       4     0.0352     0.0415
      5+     0.0667     0.0685

  !! Nelder-Mead found a HIGHER log-likelihood (-13508.562 > -13508.700).
  !! The

In [ ]:
"""
H1a tiebreaker: shifted-outcome NB  (model reuse_count - 1)
--------------------------------------------------------------------------
WHY

The zero-truncated NB is not stably identified on this data. Its dispersion parameter diverges to the boundary (alpha = 4.6e3 to 3.3e5
depending on optimizer, versus 0.26 for the ordinary NB) while the log-likelihood barely changes -- the surface is essentially flat in
alpha. As alpha -> infinity the zero-truncated NB degenerates toward the logarithmic (log-series) distribution, so the optimizer is running off
to a limiting case rather than finding an interior optimum. Standard errors and p-values from a boundary solution are not valid, so the ZTNB
cannot be used to overturn the ordinary-NB result.

THE ALTERNATIVE

Because reuse_count >= 1 by construction, define

    excess_reuse = reuse_count - 1        (>= 0, zeros legitimate)

and fit an ordinary NB to it. This respects the support without any truncation machinery and has no boundary pathology. The coefficient
describes reuse BEYOND the first observed reuse, which is the part of the outcome that actually varies.

Also fits a zero-truncated Poisson (no dispersion parameter, so no alpha to diverge) as a second, independent check.

WHAT TO CONCLUDE

  * If the shifted-NB agrees with the ordinary NB (is_permissive significant under linear age, not under log age), the paper's
    existing "weak, not robust" conclusion stands, and the truncation objection is answered.
  * If the shifted-NB shows is_permissive significant under BOTH specs, H1a should be upgraded to supported.
"""

import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.discrete.truncated_model import TruncatedLFPoisson

H1A_PATH = RESULTS / "h1a_analysis_dataframe.csv"
PREDICTOR = "is_permissive"


def run(path=H1A_PATH):
    df = pd.read_csv(path)
    if PREDICTOR not in df.columns:
        df[PREDICTOR] = (df["license_family"] == "Permissive").astype(int)
    df[PREDICTOR] = df[PREDICTOR].astype(int)
    df["age_std"] = (df["age_days"] - df["age_days"].mean()) / df["age_days"].std()
    _la = np.log(df["age_days"] + 1)
    df["log_age"] = (_la - _la.mean()) / _la.std()

    y_raw = df["reuse_count"].astype(float)
    y_shift = y_raw - 1.0

    print(f"N = {len(df):,}")
    print(f"reuse_count   : mean {y_raw.mean():.4f}, min {y_raw.min():.0f}")
    print(f"excess_reuse  : mean {y_shift.mean():.4f}, min {y_shift.min():.0f}, "
          f"zeros {int((y_shift == 0).sum()):,} ({(y_shift == 0).mean():.1%})")

    rows = []
    for age_var in ["age_std", "log_age"]:
        X = sm.add_constant(df[[PREDICTOR, age_var]])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            # Reference: ordinary NB on the raw (untruncated-assumption) outcome
            nb_raw = sm.NegativeBinomial(y_raw, X).fit(disp=False, maxiter=500)
            # Primary alternative: ordinary NB on the shifted outcome
            nb_shift = sm.NegativeBinomial(y_shift, X).fit(disp=False, maxiter=500)
            # Independent check: zero-truncated Poisson (no alpha to diverge)
            try:
                ztp = TruncatedLFPoisson(y_raw, X, truncation=0).fit(
                    disp=False, maxiter=2000)
            except Exception as e:
                ztp = None
                print(f"  [ZTP failed for {age_var}: {e}]")

        entries = [("Ordinary NB (raw y)", nb_raw), ("NB on y-1 (shifted)", nb_shift)]
        if ztp is not None:
            entries.append(("Zero-truncated Poisson", ztp))

        for name, m in entries:
            alpha = (float(m.params["alpha"]) if "alpha" in m.params.index
                     else np.nan)
            rows.append({
                "age spec": age_var,
                "model": name,
                "coef": round(float(m.params[PREDICTOR]), 4),
                "SE": round(float(m.bse[PREDICTOR]), 4),
                "IRR": round(float(np.exp(m.params[PREDICTOR])), 4),
                "p": f"{float(m.pvalues[PREDICTOR]):.4g}",
                "alpha": ("n/a" if np.isnan(alpha) else round(alpha, 4)),
                "sig@.05": bool(float(m.pvalues[PREDICTOR]) < .05),
            })

    out = pd.DataFrame(rows)
    print("\n" + out.to_string(index=False))

    print("\nSanity note: 'alpha' should stay O(0.1-10). Values in the thousands")
    print("indicate the boundary pathology that made the ZTNB unusable.")

    shifted = out[out["model"] == "NB on y-1 (shifted)"]
    n_sig = int(shifted["sig@.05"].sum())
    print(f"\nShifted-NB: is_permissive significant in {n_sig}/2 age specifications.")
    if n_sig == 2:
        print("  => Effect robust to age specification. Consider upgrading H1a.")
    elif n_sig == 1:
        print("  => Specification-dependent, consistent with the ordinary NB result.")
        print("     The paper's existing 'weak, not robust' conclusion stands.")
    else:
        print("  => No significant effect under either specification.")

    out.to_csv(RESULTS / "h1a_shifted_nb_comparison.csv", index=False)
    print("\nSaved: h1a_shifted_nb_comparison.csv")
    return out


if __name__ == "__main__":
    run()

N = 11,252
reuse_count   : mean 1.9220, min 1
excess_reuse  : mean 0.9220, min 0, zeros 7,389 (65.7%)

age spec                  model   coef     SE    IRR        p   alpha  sig@.05
 age_std    Ordinary NB (raw y) 0.0462 0.0212 1.0473  0.02938  0.2598     True
 age_std    NB on y-1 (shifted) 0.1587 0.0499 1.1720 0.001459  2.9877     True
 age_std Zero-truncated Poisson 0.0473 0.0226 1.0485  0.03644     n/a     True
 log_age    Ordinary NB (raw y) 0.0199 0.0209 1.0201   0.3402  0.2477    False
 log_age    NB on y-1 (shifted) 0.0993 0.0492 1.1044  0.04361  2.8941     True
 log_age Zero-truncated Poisson 0.0632 0.0222 1.0653 0.004408     n/a     True

Sanity note: 'alpha' should stay O(0.1-10). Values in the thousands
indicate the boundary pathology that made the ZTNB unusable.

Shifted-NB: is_permissive significant in 2/2 age specifications.
  => Effect robust to age specification. Consider upgrading H1a.

Saved: h1a_shifted_nb_comparison.csv
